In [ ]:
%pip install openai requests python-dotenv

In [8]:
import os
import json
import requests
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY")
)

MODEL="llama-3.1-8b-instant"

In [ ]:

def get_weather(latitude: float, longitude: float) -> str:
    """
    url for live weather data from open-meteo.com, which does not require api key
    """
    url = f"https://api.open-meteo.com/v1/forecast?latitude=48.8566&longitude=2.3522&current=temperature_2m"
    try:
        response = requests.get(url)
        data = response.json()
        temp = data['current']['temperature_2m']
        return f"The current temperature at coordinates ({latitude}, {longitude}) is {temp}°C."
    except Exception as e:
        return f"Error reaching local weather system layer: {str(e)}"

def main():
    # json schema
    
    tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given latitude and longitude location",
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": {
                        "type": "number",
                        "description": "The latitude coordinate of the city"
                    },
                    "longitude": {
                        "type": "number",
                        "description": "The longitude coordinate of the city"
                    }
                },
                "required": ["latitude", "longitude"]
            }
        }
    }
]

    # initiate conversation
    messages = [
        {"role": "user", "content": "What's the weather like in Delhi today?"}
    ]

    print(f"User Prompt: {messages[0]['content']}\n")
    print("--- Turn 1: Dispatching Payload to Groq Engine ---")

    # send initial request
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools_schema,
        tool_choice="auto"
    )

    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls

    # check if model needs to call tools
    if tool_calls:
        print("💡 [MODEL DECISION]: Function execution requested by Groq.")
        
        # append model reponse to history for context
        messages.append(response_message)

        # process all tool calls
        for tool_call in tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            
            print(f"-> Model requested function: '{function_name}'")
            print(f"-> Extracted Parameters: {function_args}")

            if function_name == "get_weather":
                # extract type-casted coordinates 
                lat = function_args.get("latitude")
                lon = function_args.get("longitude")
                
                # execute real local function
                tool_output_result = get_weather(latitude=lat, longitude=lon)
                print(f"🔄 Executed local function tool outcome: {tool_output_result}\n")

                # append function output to history
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": tool_output_result
                })

        # send full context back to groq for summary
        print("--- Turn 2: Requesting Final Conversational Synthesis ---")
        final_response = client.chat.completions.create(
            model=MODEL,
            messages=messages
        )

        final_answer = final_response.choices[0].message.content
        print(f"\nFinal Assistant Response:\n{final_answer}")
        
    else:
        print("Assistant Response (No tools triggered):")
        print(response_message.content)

if __name__ == "__main__":
    main()

User Prompt: What's the weather like in Delhi today?

--- Turn 1: Dispatching Payload to Groq Engine ---
💡 [MODEL DECISION]: Function execution requested by Groq.
-> Model requested function: 'get_weather'
-> Extracted Parameters: {'latitude': 28.7041, 'longitude': 77.1025}
🔄 Executed local function tool outcome: The current temperature at coordinates (28.7041, 77.1025) is 36.8°C.

--- Turn 2: Requesting Final Conversational Synthesis ---

Final Assistant Response:
Please note that this function does not provide real-time current information, but rather estimates based on the last known data.
